# Breakers, Fuses, Reclosers, and other Switching Devices

----

There are eight different kinds of Switch supported in the CIM, all with zero impedance. They will all behave the same in power flow analysis, and all would require many more attributes than are defined in CIM to support protection analysis. The classes of Switch defined in CIM are divided into two categories depending on whether the device is capable of interrupting fault currents, as shown in Figure 19 below. Multiple switches in a single vault or pad-mounted switchgear can be grouped using the CompositeSwitch class.

In [1]:
import os
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [2]:
diagram_text = utils.get_mermaid([cim.Switch, cim.SwitchPhase, cim.Sectionaliser, cim.Jumper, cim.Fuse, cim.Disconnector, cim.GroundDisconnector, cim.ProtectedSwitch, cim.Breaker, cim.Recloser, cim.LoadBreakSwitch])
Mermaid(diagram_text)

Devices inheriting from ProtectedSwitch include:
1)	LoadBreakSwitch: a generic mechanical switching device capable of breaking current under normal operating conditions
2)	Recloser: a pole-mounted fault interrupter with built-in phase/ground relays and CT
3)	Breaker: a substation circuit breaker


Other switching devices include:
1)	Sectionaliser: a switching device locked open to isolate a faulted section, with or without any load-breaking capability (note that the class is spelled with a letter “s” rather than “z”).
2)	Fuse: a protective fuse used on lines, transformers, and other equipment. Note that Fuse does not inherit from ProtectedSwitch because fuses do not use a protection relay.
3)	Jumper: A short section of conductor with negligible impedance that is manually installed by field crews 
4)	Disconnector: a manually-operated or motor-operated switch used to isolate circuits or equipment in a substation, not capable of breaking any significant current
5)	GroundDisconnector: a manually-operated or motor-operated switch used to ground equipment for maintenance in a substation.


In unbalanced circuits, individual phases of a switch are modeled as attributes of the SwitchPhase class. The use cases for SwitchPhase include 1) single-phase, two-phase, and secondary switches, 2) one or two conductors open in a three-phase switch or 3) transpositions, in which case phaseSide1 and phaseSide2 would be different. RatedCurrent may be different among the phases, e.g., individual fuses on the same pole may have different ratings. Note that the Switch class uses the Boolean attribute Switch:open (with 0 meaning closed and 1 open), while the SwitchPhase class uses the attribute SwitchPhase.closed.

Some examples are discussed below.

In [3]:
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel

In [4]:
network = FeederModel(connection=XMLFile(filename='../sample_models/ieee13.xml'), container=None)

Example 1: How many switches of all types are in there in the feeder?

In [5]:
# Every switching device type, including the Switch and ProtectedSwitch base classes
switch_classes = [cim.Breaker, cim.Fuse, cim.Recloser, cim.Sectionaliser,
                  cim.LoadBreakSwitch, cim.Switch]

results = []
total = 0
for switch_class in switch_classes:
    switches = network.list_by_class(switch_class)
    if switches:
        results.append({'type': switch_class.__name__, 'qty': len(switches)})
        total += len(switches)

print(results)
print(total)

[{'type': 'Breaker', 'qty': 1}, {'type': 'Fuse', 'qty': 1}, {'type': 'Recloser', 'qty': 1}, {'type': 'LoadBreakSwitch', 'qty': 2}]
5


Example 2: What phases are associated with switch with mRID 2858B6C2-0886-4269-884C-06FA8B887319?

In [6]:
switch = network.get_object(mRID='2858B6C2-0886-4269-884C-06FA8B887319')

results = []
for switch_phase in switch.SwitchPhase:
    results.append(switch_phase.phaseSide1)
    results.append(switch_phase.phaseSide2)

print(results)

[<SinglePhaseKind.C: 'C'>, <SinglePhaseKind.C: 'C'>]


Example 3: Find the mRIDs of all the single-phase switches on phase C?

In [7]:
results = set()

for switch_phase in network.list_by_class(cim.SwitchPhase):
    if cim.SinglePhaseKind.C in (switch_phase.phaseSide1, switch_phase.phaseSide2):
        results.add(switch_phase.Switch.mRID)

print(results)

{'2858B6C2-0886-4269-884C-06FA8B887319'}


Example 4: Provide the phases of discrete measurements of all breakers

In [8]:
results = []

for breaker in network.list_by_class(cim.Breaker):
    for measurement in breaker.Measurements:
        if isinstance(measurement, cim.Discrete):
            results.append(str(measurement.phases))

print(results)

['PhaseCode.A', 'PhaseCode.B', 'PhaseCode.C']


Example 5: List whether each switch is open or closed for all types of switches

In [9]:
switch_classes = [cim.Breaker, cim.Fuse, cim.Recloser, cim.Sectionaliser,
                  cim.LoadBreakSwitch, cim.Switch]

results = []
for switch_class in switch_classes:
    for switch in network.list_by_class(switch_class):
        # Switch.open is True when the device is open, False when closed
        results.append({'name': switch.name, 'position': switch.open})

print(results)

[{'name': 'brkr1', 'position': False}, {'name': 'fuse1', 'position': False}, {'name': 'rec1', 'position': False}, {'name': 'sect1', 'position': False}, {'name': '671692', 'position': False}]


Example 6: What is the current rating of fuse with mRID 43EF8365-F932-409B-A51E-FBED3F6DFFAA?

In [10]:
fuse = network.get_object(mRID='43EF8365-F932-409B-A51E-FBED3F6DFFAA')

print(fuse.ratedCurrent)

100.0


Example 7: List the measurement mRIDs of all switches of all types

In [11]:
switch_classes = [cim.Breaker, cim.Fuse, cim.Recloser, cim.Sectionaliser,
                  cim.LoadBreakSwitch, cim.Switch]

results = []
for switch_class in switch_classes:
    for switch in network.list_by_class(switch_class):
        for measurement in switch.Measurements:
            results.append(measurement.mRID)

print(results)

['93b441c7-6626-4a50-af02-27e2cdb847bf', '9a379d4c-b434-4743-bd8f-87add6d7df92', '8b230dfe-3188-43b1-8e93-8839ad6c13f7', 'db91a191-bf02-4b08-9cde-54857c223d2e', 'ef2fa8e2-74cc-44b0-85ee-b8186e6cee54', '3e7ee924-ff6e-4640-8c09-38d76d447cd6', '73959d90-d823-48b8-be07-72ba0247e67a', '1b70f895-30eb-43d0-9e62-2d4bd4f34677', 'ffb8967e-61fd-461e-9ffe-21795440a52e', '34b95861-e956-498e-91f5-0b18a1c1a829', '6452744d-b9bd-4cda-9430-16cacd630101', 'e1d8c475-0c39-4a09-a630-2921a5760c1d', 'f396638c-63ca-4532-8df6-8a182d800647', '777cdb29-c06c-458b-9915-6b2b8875c43a', 'eeae8bd7-c748-4a8f-b1e0-a30e979d58de', '02efc0b9-5784-4b28-9547-9394a377947d', 'cbb63b9c-ce44-443d-a56b-162c2b0222c9', '20fccd4b-15a6-4c6d-9970-0d870f503cdb', '05272dc9-5f3a-4583-839b-f5acb49557bb', 'f6eae109-55d6-4b41-b2a5-39f3e7ba60fe']
